# Subperfiles anidados (árbol de arriba hacia abajo)

El árbol del notebook 07, construido de abajo hacia arriba, resultó inestable (ARI 0.33): sobre
un continuo, los cortes cambian con cada remuestreo. Aquí se invierte la dirección:

1. **Perfiles:** las 6 clases latentes del notebook 08 (ARI bootstrap 0.99), que combinan nivel
   y forma de la trayectoria. Alternativa: los 4 perfiles del notebook 04 (ARI 0.74).
2. **Subperfiles:** KMeans ponderado dentro de cada perfil. El número de subperfiles de cada
   perfil es el mayor que resulta estable en bootstrap.

Para microtargeting, cada subperfil se describe con las entidades, personas-tipo (sexo x edad) y
municipios donde esta sobrerrepresentado.

## 1. Configuración

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

## 2. Parametros y rutas

- `ORIGEN_PERFILES`: `"clases_latentes"` (notebook 08) o `"base_04"` (notebook 04).
- `SUB_VARIABLES`: espacio en que se subdivide cada perfil (promedio, brecha y tendencia).
- `K_SUB_FIJO`: fija a mano el número de subperfiles por perfil.

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DIR_PREPROCESSED = ROOT / "DATA" / "PREPROCESSED"

ELECCIONES = ["P2009", "P2012", "P2015", "P2018", "P2021", "P2024"]
CORTES_EDAD = [17, 24, 29, 39, 49, 59, 69, 79, 200]
ETIQUETAS_EDAD = ["18-24", "25-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
LLAVES = ["EDOCVE", "SECCIÓN", "SEXO", "GRUPO_EDAD"]
BASE = ["promedio", "brecha_pres", "tendencia"]

ORIGEN_PERFILES = "clases_latentes"
SUB_VARIABLES = BASE
K_SUB_RANGO = range(2, 6)
K_SUB_FIJO = None  # ej. {0: 3, 1: 2, 2: 3, 3: 3}
MIN_PCT_SUB = 2.0  # % de la población total
ARI_ESTABLE = 0.8

N_EVAL = 10_000
N_SIL = 5_000
N_BOOT = 3
MIN_PERSONAS_ENTIDAD = 20_000
MIN_PERSONAS_TIPO = 20_000
MIN_PERSONAS_MPIO = 1_000
RANDOM_STATE = 42

COLORES_PERFIL = ["#3288bd", "#66c2a5", "#fee08b", "#fdae61", "#f46d43", "#d53e4f",
                  "#9e0142", "#5e4fa2"]
EDONOM = {
    1: "Aguascalientes", 2: "Baja California", 3: "Baja California Sur",
    4: "Campeche", 5: "Coahuila", 6: "Colima", 7: "Chiapas", 8: "Chihuahua",
    9: "Ciudad de México", 10: "Durango", 11: "Guanajuato", 12: "Guerrero",
    13: "Hidalgo", 14: "Jalisco", 15: "México", 16: "Michoacán", 17: "Morelos",
    18: "Nayarit", 19: "Nuevo Leon", 20: "Oaxaca", 21: "Puebla", 22: "Queretaro",
    23: "Quintana Roo", 24: "San Luis Potosi", 25: "Sinaloa", 26: "Sonora",
    27: "Tabasco", 28: "Tamaulipas", 29: "Tlaxcala", 30: "Veracruz",
    31: "Yucatan", 32: "Zacatecas",
}

### 2.1 Funciones auxiliares

Resumenes ponderados por peso y R2 ponderado (proporción de varianza explicada por los grupos).

In [ ]:
def media_ponderada(df, col, por):
    grupos = [df[c] for c in por]
    wx = (df[col] * df["peso"]).groupby(grupos).sum()
    return wx / df["peso"].where(df[col].notna()).groupby(grupos).sum()


def dominante(df, por, col):
    t = df.groupby(por + [col])["peso"].sum()
    return t.groupby(level=por).idxmax().map(lambda x: x[-1])


def r2_ponderado(Z, labels, w):
    wl = np.bincount(labels, weights=w)
    centros = np.stack([np.bincount(labels, weights=w * Z[:, j]) for j in range(Z.shape[1])], axis=1)
    centros = centros / wl[:, None]
    total = (w[:, None] * (Z - np.average(Z, axis=0, weights=w)) ** 2).sum()
    dentro = (w[:, None] * (Z - centros[labels]) ** 2).sum()
    return 1 - dentro / total


def resumen(df, por):
    personas = df.groupby(por)["peso"].sum()
    out = pd.DataFrame({"personas": personas, "pct_pob": personas / df["peso"].sum() * 100})
    for col in ["promedio", "brecha_pres", "tendencia", "desviación", "P_PJ2025", "MUJER", "RURAL_MIXTA"]:
        out[col] = media_ponderada(df, col, por)
    out[["MUJER", "RURAL_MIXTA"]] *= 100
    out["edad_dominante"] = dominante(df, por, "GRUPO_EDAD")
    out["region_dominante"] = dominante(df, por, "REGIÓN")
    return out.round(1)

## 3. Carga de datos

### 3.1 Celdas, peso, PJ 2025 y municipio

Celdas del notebook 03 con peso = SV + NV de 2024. Se conservan las celdas sin tasa PJ 2025
(Campeche no tiene datos de la elección judicial); los promedios PJ usan solo celdas con dato.

In [ ]:
celdas = pd.read_parquet(DIR_PREPROCESSED / "celdas_historial.parquet")
celdas["GRUPO_EDAD"] = celdas["GRUPO_EDAD"].astype(str)

fed24 = pd.read_parquet(
    DIR_PREPROCESSED / "conteos_federal_2009_2024.parquet",
    columns=["EDOCVE", "SECCIÓN", "SEXO", "EDAD", "AELEC", "SV", "NV", "MPIOCVE", "MPIONOM"],
    filters=[("AELEC", "==", 2024)],
)
fed24["GRUPO_EDAD"] = pd.cut(fed24["EDAD"], bins=CORTES_EDAD, labels=ETIQUETAS_EDAD)
fed24["VOTANTES"] = fed24["SV"] + fed24["NV"]
pesos = (
    fed24.groupby(LLAVES, observed=True)["VOTANTES"]
    .sum()
    .reset_index()
    .rename(columns={"VOTANTES": "peso"})
)
pesos["GRUPO_EDAD"] = pesos["GRUPO_EDAD"].astype(str)
sec_mpio = (
    fed24.groupby(["EDOCVE", "SECCIÓN"])
    .agg(MPIOCVE=("MPIOCVE", "first"), MPIONOM=("MPIONOM", "first"))
    .reset_index()
)
del fed24

secciones = pd.read_parquet(
    DIR_PREPROCESSED / "secciones_historial.parquet", columns=["EDOCVE", "SECCIÓN", "P_PJ2025"]
)
celdas = (
    celdas.merge(pesos, on=LLAVES, how="left")
    .merge(secciones, on=["EDOCVE", "SECCIÓN"], how="left")
    .merge(sec_mpio, on=["EDOCVE", "SECCIÓN"], how="left")
    .dropna(subset=["peso"])
    .reset_index(drop=True)
)
del pesos, secciones, sec_mpio

celdas["MUJER"] = (celdas["SEXO"] == 1).astype(float)
celdas["RURAL_MIXTA"] = (celdas["TIPOSEC"].astype(str) != "U").astype(float)
celdas["persona_tipo"] = celdas["SEXO"].map({0: "Hombre", 1: "Mujer"}) + " " + celdas["GRUPO_EDAD"]
w = celdas["peso"].to_numpy(dtype=float)
idx_eval = np.random.default_rng(RANDOM_STATE).choice(len(celdas), size=N_EVAL, replace=False, p=w / w.sum())
print(f"Celdas: {len(celdas):,}  peso: {w.sum():,.0f}")

### 3.2 Perfiles

- `clases_latentes`: columna `clase` de `celdas_clases_latentes.parquet` (notebook 08).
- `base_04`: columna `base_04` de `celdas_perfiles_candidatos.parquet` (reajuste del modelo del
  notebook 04 en el notebook 05).

En ambos casos se renumeran por promedio de participación (0 = menor).

In [ ]:
ORIGENES = {
    "clases_latentes": ("celdas_clases_latentes.parquet", "clase"),
    "base_04": ("celdas_perfiles_candidatos.parquet", "base_04"),
}
archivo, columna = ORIGENES[ORIGEN_PERFILES]
origen = pd.read_parquet(DIR_PREPROCESSED / archivo, columns=LLAVES + [columna])
n0 = len(celdas)
celdas = celdas.merge(origen.rename(columns={columna: "perfil_origen"}), on=LLAVES, how="inner")
print(f"Celdas sin perfil de origen (excluidas): {n0 - len(celdas):,}")
w = celdas["peso"].to_numpy(dtype=float)
idx_eval = np.random.default_rng(RANDOM_STATE).choice(len(celdas), size=N_EVAL, replace=False, p=w / w.sum())

orden = media_ponderada(celdas, "promedio", ["perfil_origen"]).sort_values().index
celdas["perfil"] = celdas["perfil_origen"].map({old: new for new, old in enumerate(orden)}).astype(int)
celdas = celdas.drop(columns="perfil_origen")
K_PERFILES = celdas["perfil"].nunique()
print(f"Perfiles: {ORIGEN_PERFILES} ({K_PERFILES})")
resumen(celdas, ["perfil"])

### 3.3 Espacio de clustering

Estandarizacion ponderada por peso. `Z_base` reproduce el espacio de los perfiles y `Z` es el
espacio de subdivision (iguales con la configuración por defecto).

In [ ]:
def estandarizar(cols):
    X = celdas[cols].to_numpy(dtype=float)
    return StandardScaler().fit(X, sample_weight=w).transform(X)


Z_base = estandarizar(BASE)
Z = estandarizar(SUB_VARIABLES)
perfil = celdas["perfil"].to_numpy()
print(f"Espacio de subdivision: {SUB_VARIABLES}")

## 4. Estabilidad de los perfiles

Si los perfiles no fueran estables, no tendría sentido anidar subperfiles en ellos. Las clases
latentes ya se evaluaron en el notebook 08 (bootstrap de Poisson, ARI 0.99). Para `base_04` se
repite aquí el bootstrap de Poisson (peso x Poisson(1)) del KMeans de 4 perfiles.

In [ ]:
if ORIGEN_PERFILES == "base_04":
    aris = []
    for b in range(N_BOOT):
        w_b = w * np.random.default_rng(1000 + b).poisson(1.0, size=len(w))
        lab_b = KMeans(K_PERFILES, n_init=5, random_state=1000 + b).fit(Z_base, sample_weight=w_b).labels_
        aris.append(adjusted_rand_score(perfil[idx_eval], lab_b[idx_eval]))
    print(f"ARI de perfiles: medio {np.mean(aris):.3f}, mínimo {np.min(aris):.3f}")
else:
    print("Estabilidad de las clases latentes: ver notebook 08 (sección 7.1).")

## 5. Subperfiles

### 5.1 Número de subperfiles por perfil

Para cada perfil y k = 2-5: KMeans ponderado sobre las celdas del perfil, silhouette sobre una
muestra ponderada de 5k celdas del perfil, R2 dentro del perfil, tamaño del subperfil más chico
(% de la población total) y ARI medio en 3 bootstraps.

In [ ]:
def muestra_perfil(mask, seed):
    idx = np.flatnonzero(mask)
    return np.random.default_rng(seed).choice(
        idx, size=min(N_SIL, len(idx)), replace=False, p=w[idx] / w[idx].sum()
    )


filas, modelos_sub = [], {}
for p in range(K_PERFILES):
    mask = perfil == p
    idx_s = muestra_perfil(mask, RANDOM_STATE + p)
    for k in K_SUB_RANGO:
        km = KMeans(k, n_init=10, random_state=RANDOM_STATE).fit(Z[mask], sample_weight=w[mask])
        modelos_sub[(p, k)] = km
        lab_s = km.predict(Z[idx_s])
        aris = []
        for b in range(N_BOOT):
            w_b = w[mask] * np.random.default_rng(1000 + b).poisson(1.0, size=mask.sum())
            km_b = KMeans(k, n_init=3, random_state=1000 + b).fit(Z[mask], sample_weight=w_b)
            aris.append(adjusted_rand_score(lab_s, km_b.predict(Z[idx_s])))
        tam = np.bincount(km.labels_, weights=w[mask], minlength=k) / w.sum() * 100
        filas.append({
            "perfil": p, "k": k,
            "silhouette": silhouette_score(Z[idx_s], lab_s),
            "r2": r2_ponderado(Z[mask], km.labels_, w[mask]),
            "pct_min_total": tam.min(),
            "ari_boot": np.mean(aris),
        })

seleccion = pd.DataFrame(filas)
seleccion.round(3)

### 5.2 Elección

Por perfil, el mayor k que es estable (ARI >= 0.8) y cuyo subperfil más chico tiene al menos 2%
de la población: el mayor detalle que sigue siendo reproducible. Si ningún k cumple, el de mayor
silhouette.

In [ ]:
if K_SUB_FIJO is None:
    K_SUB = {}
    for p, d in seleccion.groupby("perfil"):
        ok = d[(d["ari_boot"] >= ARI_ESTABLE) & (d["pct_min_total"] >= MIN_PCT_SUB)]
        K_SUB[p] = int(ok["k"].max()) if len(ok) else int(d.loc[d["silhouette"].idxmax(), "k"])
else:
    K_SUB = K_SUB_FIJO
print(f"Subperfiles por perfil: {K_SUB}  (total {sum(K_SUB.values())})")

### 5.3 Asignación

Los subperfiles se numeran dentro de su perfil por promedio de participación (ej. 2.1, 2.2).

In [ ]:
sub_k = np.full(len(celdas), -1)
for p, k in K_SUB.items():
    sub_k[perfil == p] = modelos_sub[(p, k)].labels_
celdas["sub_k"] = sub_k

prom = (
    media_ponderada(celdas, "promedio", ["perfil", "sub_k"])
    .rename("prom")
    .reset_index()
    .sort_values("prom")
)
prom["subperfil"] = prom["perfil"].astype(str) + "." + (prom.groupby("perfil").cumcount() + 1).astype(str)
celdas = celdas.merge(prom[["perfil", "sub_k", "subperfil"]], on=["perfil", "sub_k"], how="left")

estab = seleccion.set_index(["perfil", "k"]).loc[list(K_SUB.items()), ["silhouette", "ari_boot"]]
estab.round(3)

## 6. Caracterización

### 6.1 Resumen por subperfil

In [ ]:
tabla_sub = resumen(celdas, ["perfil", "subperfil"])
tabla_sub

### 6.2 Trayectorias

Un panel por perfil con la trayectoria ponderada de cada subperfil.

In [ ]:
anios = [2009, 2012, 2015, 2018, 2021, 2024]
tray = pd.DataFrame({e: media_ponderada(celdas, e, ["perfil", "subperfil"]) for e in ELECCIONES})

ncols = min(K_PERFILES, 3)
nrows = int(np.ceil(K_PERFILES / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharey=True, squeeze=False)
axes = axes.ravel()
for ax in axes[K_PERFILES:]:
    ax.set_visible(False)
for p, ax in enumerate(axes[:K_PERFILES]):
    for (pp, s), fila in tray.loc[[p]].iterrows():
        ax.plot(anios, fila.to_numpy(), "o-", label=f"{s} ({tabla_sub.loc[(pp, s), 'pct_pob']:.1f}%)")
    ax.set_title(f"Perfil {p}", color=COLORES_PERFIL[p], fontweight="bold")
    ax.set_xticks(anios, [str(a)[2:] for a in anios])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
for ax in axes[::ncols]:
    ax.set_ylabel("% participación (ponderada)")
plt.tight_layout()
plt.show()

### 6.3 Mapa de subperfiles

Cada burbuja es un subperfil, con tamaño proporcional a las personas que representa.

In [ ]:
t = tabla_sub.reset_index()

fig, ax = plt.subplots(figsize=(9, 6))
for p, d in t.groupby("perfil"):
    ax.scatter(d["promedio"], d["brecha_pres"], s=d["pct_pob"] * 60, color=COLORES_PERFIL[p],
               alpha=0.8, edgecolor="#555", linewidth=0.5, label=f"Perfil {p}")
    for _, r in d.iterrows():
        ax.annotate(r["subperfil"], (r["promedio"], r["brecha_pres"]), ha="center", va="center", fontsize=8)
ax.set_xlabel("Promedio de participación 2009-2024 (%)")
ax.set_ylabel("Brecha presidencial (puntos)")
ax.set_title("Subperfiles")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Microtargeting

### 7.1 Índice de sobrerrepresentación

Para cada subperfil y cada categoria (entidad o persona-tipo):
`índice = (% de la categoria que pertenece al subperfil) / (% nacional del subperfil) x 100`.
100 = igual que el promedio nacional; 200 = el doble. Se exige un mínimo de personas para no
destacar categorias diminutas.

In [ ]:
def sobrerrepresentacion(df, dim, min_personas, top=3):
    t = df.groupby(["subperfil", dim])["peso"].sum().rename("personas").reset_index()
    t["pct_en_categoria"] = t["personas"] / t.groupby(dim)["personas"].transform("sum") * 100
    pct_nacional = df.groupby("subperfil")["peso"].sum() / df["peso"].sum() * 100
    t["índice"] = t["pct_en_categoria"] / t["subperfil"].map(pct_nacional) * 100
    t = t[t["personas"] >= min_personas]
    return t.sort_values("índice", ascending=False).groupby("subperfil").head(top).sort_values(
        ["subperfil", "índice"], ascending=[True, False]
    )


def como_texto(t, etiqueta):
    txt = t[etiqueta] + " (" + t["índice"].round(0).astype(int).astype(str) + ")"
    return txt.groupby(t["subperfil"]).agg(", ".join)


ent = sobrerrepresentacion(celdas.assign(entidad=celdas["EDOCVE"].map(EDONOM)), "entidad", MIN_PERSONAS_ENTIDAD)
tipo = sobrerrepresentacion(celdas, "persona_tipo", MIN_PERSONAS_TIPO)

micro = tabla_sub[["pct_pob", "promedio", "brecha_pres", "tendencia"]].reset_index()
micro["entidades"] = micro["subperfil"].map(como_texto(ent, "entidad"))
micro["personas_tipo"] = micro["subperfil"].map(como_texto(tipo, "persona_tipo"))
pd.set_option("display.max_colwidth", 80)
micro

### 7.2 Municipios

Subperfil dominante de cada municipio con al menos 1,000 personas y su porcentaje. Se cuentan los
municipios por subperfil dominante.

In [ ]:
llave_mpio = ["EDOCVE", "MPIOCVE"]
mpio = celdas.groupby(llave_mpio + ["MPIONOM", "subperfil"])["peso"].sum().reset_index()
mpio["personas_mpio"] = mpio.groupby(llave_mpio)["peso"].transform("sum")
mpio["pct_sub"] = mpio["peso"] / mpio["personas_mpio"] * 100
municipios = mpio.loc[mpio.groupby(llave_mpio)["peso"].idxmax()]
municipios = municipios[municipios["personas_mpio"] >= MIN_PERSONAS_MPIO].copy()
municipios["entidad"] = municipios["EDOCVE"].map(EDONOM)
municipios = municipios.rename(columns={"peso": "personas_sub"})

print(f"Municipios con al menos {MIN_PERSONAS_MPIO:,} personas: {len(municipios):,}")
municipios["subperfil"].value_counts().sort_index().rename("municipios")

In [ ]:
municipios.sort_values("pct_sub", ascending=False).head(15)[
    ["entidad", "MPIONOM", "personas_mpio", "subperfil", "pct_sub"]
].round(1)

## 8. Comparación con modelos previos

ARI contra los perfiles del notebook 04, el árbol del 07 y las clases latentes del 08, si existen.
Con `ORIGEN_PERFILES = "clases_latentes"`, el ARI de perfil contra clase es 1 por construcción.

In [ ]:
comparar = {
    "celdas_perfiles_candidatos.parquet": {"perfil": "base_04", "subperfil": "base_04"},
    "celdas_jerarquia.parquet": {"perfil": "perfil", "subperfil": "subperfil"},
    "celdas_clases_latentes.parquet": {"perfil": "clase", "subperfil": "clase"},
}
for archivo, cols in comparar.items():
    ruta = DIR_PREPROCESSED / archivo
    if not ruta.exists():
        print(f"No existe {archivo}")
        continue
    otro = pd.read_parquet(ruta, columns=LLAVES + sorted(set(cols.values())))
    otro = celdas[LLAVES].merge(otro, on=LLAVES, how="left")
    for propio, ajeno in cols.items():
        ari = adjusted_rand_score(celdas[propio].astype(str).to_numpy()[idx_eval],
                                  otro[ajeno].astype(str).fillna("sin dato").to_numpy()[idx_eval])
        print(f"ARI {propio} vs {archivo} [{ajeno}]: {ari:.3f}")

## 9. Exportacion

No se sobrescribe nada de los notebooks 04-08.

In [ ]:
salidas = {
    "celdas_subperfiles.parquet": celdas[LLAVES + ["peso", "perfil", "subperfil"]],
    "subperfiles_resumen.csv": tabla_sub,
    "subperfiles_microtargeting.csv": micro,
    "subperfiles_entidades.csv": ent,
    "subperfiles_personas_tipo.csv": tipo,
    "subperfiles_municipios.csv": municipios,
}
for nombre, df in salidas.items():
    ruta = DIR_PREPROCESSED / nombre
    if nombre.endswith(".parquet"):
        df.to_parquet(ruta, index=False)
    else:
        df.to_csv(ruta, index=nombre == "subperfiles_resumen.csv")
    print(f"{nombre}: {ruta.stat().st_size / 1024:.0f} KB")